# Notebook 14 — EKF-SLAM State Augmentation

This notebook focuses on the **EKF-SLAM state augmentation step**: how we grow the state vector when a new landmark is first observed.

We keep this beginner-friendly and visual, with small worlds and short runs.

## Learning goals
- Understand the SLAM state vector structure: `[x, y, theta, l1x, l1y, l2x, l2y, ...]`.
- Read the covariance matrix in block form.
- See how and why we add a new landmark to state + covariance.
- Build intuition for why cross-covariances matter.
- Run a tiny EKF-SLAM simulation and inspect innovation statistics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from kiss_slam.ekf_slam import EKFSLAM
from kiss_slam.models.motion import DifferentialDriveMotionModel
from kiss_slam.models.measurement import RangeBearingMeasurementModel
from kiss_slam.sim.world import World2D
from kiss_slam.sim.simulator import Simulator2D, SimConfig
from kiss_slam.types import EKFSLAMConfig, Measurement, ControlInput

np.random.seed(7)


## 1) State vector structure (what lives in EKF-SLAM state?)

In EKF-SLAM we keep one long vector:

\[
\mu = [x, y, \theta, l_{1x}, l_{1y}, l_{2x}, l_{2y}, ...]^T
\]

- The first 3 values are the robot pose.
- Every landmark contributes 2 values `(x, y)`.

So with `N` landmarks, state size is `3 + 2N`.


In [ ]:
def make_state(robot_pose, landmarks_xy):
    """Build a toy EKF-SLAM state vector for display."""
    robot_pose = np.asarray(robot_pose, dtype=float)
    landmarks_xy = np.asarray(landmarks_xy, dtype=float).reshape(-1)
    return np.concatenate([robot_pose, landmarks_xy])

mu_demo = make_state(
    robot_pose=[1.0, -0.5, 0.2],
    landmarks_xy=[[2.0, 1.0], [4.5, -1.0]],
)

print("state vector:", mu_demo)
print("state dimension:", mu_demo.size, "= 3 + 2*2")


## 2) Covariance block structure (beginner view)

Think of covariance as a matrix that answers:
- "How uncertain is each variable?" (diagonal terms)
- "How much do two variables move together?" (off-diagonal terms)

For EKF-SLAM, we can partition covariance as blocks:

\[
\Sigma =
\begin{bmatrix}
\Sigma_{rr} & \Sigma_{rl} \\
\Sigma_{lr} & \Sigma_{ll}
\end{bmatrix}
\]

- `Σ_rr`: robot uncertainty (3x3)
- `Σ_ll`: landmark uncertainty (2N x 2N)
- `Σ_rl`, `Σ_lr`: **cross-covariances** linking robot and landmarks


In [ ]:
# Build a toy covariance with visible block structure.
Sigma_demo = np.zeros((7, 7), dtype=float)  # 3 robot + 2 landmarks (4 dims)

# robot block
Sigma_demo[:3, :3] = np.array([
    [0.20, 0.05, -0.02],
    [0.05, 0.30, 0.01],
    [-0.02, 0.01, 0.10],
])

# landmark block
Sigma_demo[3:, 3:] = np.array([
    [0.80, 0.10, 0.30, 0.00],
    [0.10, 0.70, 0.00, 0.25],
    [0.30, 0.00, 0.90, 0.12],
    [0.00, 0.25, 0.12, 0.95],
])

# cross blocks (robot <-> landmarks)
Sigma_demo[:3, 3:] = np.array([
    [0.20, 0.00, 0.15, 0.03],
    [0.04, 0.25, 0.01, 0.11],
    [0.01, 0.03, 0.05, 0.08],
])
Sigma_demo[3:, :3] = Sigma_demo[:3, 3:].T

plt.figure(figsize=(5, 4))
plt.imshow(Sigma_demo, cmap="viridis")
plt.colorbar(label="covariance value")
plt.title("Toy EKF-SLAM covariance with blocks")
plt.xlabel("state index")
plt.ylabel("state index")
plt.tight_layout()
plt.show()


## 3) Interactive walk-through: robot-only EKF, then add first landmark

We start with an EKF-SLAM object that has only robot pose in state.
Then we feed one landmark observation and watch state augmentation happen.


In [ ]:
slam = EKFSLAM(
    motion_model=DifferentialDriveMotionModel(),
    measurement_model=RangeBearingMeasurementModel(),
    config=EKFSLAMConfig(),
)

print("Initial state dimension:", slam.state.size)
print("Initial covariance shape:", slam.covariance.shape)

# Robot-only predict steps (no measurements yet)
for _ in range(5):
    slam.predict(u=ControlInput(v=0.6, w=0.1), dt=0.1)

print("After robot-only predicts:")
print("State dimension:", slam.state.size)
print("Covariance shape:", slam.covariance.shape)
print("Estimated robot pose:", np.round(slam.robot_pose(), 3))


In [ ]:
cov_before = slam.covariance.copy()

# First-ever landmark observation -> triggers state augmentation
first_obs = Measurement(landmark_id=42, range_m=4.0, bearing_rad=np.deg2rad(25.0))
slam.update([first_obs])

cov_after = slam.covariance.copy()
print("After first landmark observation:")
print("State dimension:", slam.state.size)
print("Covariance shape:", slam.covariance.shape)
print("Known landmarks:", sorted(slam.landmark_states().keys()))
print("State vector:", np.round(slam.state, 3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(cov_before, cmap="magma")
axes[0].set_title("Before augmentation (robot only)")
axes[0].set_xlabel("index")
axes[0].set_ylabel("index")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(cov_after, cmap="magma")
axes[1].set_title("After augmentation (robot + landmark)")
axes[1].set_xlabel("index")
axes[1].set_ylabel("index")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()


## 4) Why cross-covariances matter

When a landmark is initialized, its uncertainty depends on robot pose uncertainty.
So new landmark covariance should include:

- robot↔landmark cross terms
- landmark↔landmark cross terms (indirectly, through robot)

If we force those cross terms to zero, we pretend robot and landmark are independent.
That makes later EKF updates less informative (and often inconsistent).


In [ ]:
def ekf_update(mu, Sigma, z, z_hat, H, R):
    """One EKF update step for comparison experiments."""
    innov = z - z_hat
    S = H @ Sigma @ H.T + R
    K = Sigma @ H.T @ np.linalg.inv(S)
    mu_new = mu + K @ innov
    Sigma_new = (np.eye(Sigma.shape[0]) - K @ H) @ Sigma
    return mu_new, Sigma_new

# Build H for landmark 42 (which starts at index 3)
start = 3
robot = slam.state[:3]
landmark = slam.state[start:start+2]
hr, hl = slam.measurement_model.jacobians(robot, landmark)
H = np.zeros((2, slam.state.size))
H[:, :3] = hr
H[:, start:start+2] = hl

# Fake small innovation to apply a correction
z_hat = slam.measurement_model.predict(robot, landmark)
z = z_hat + np.array([0.3, np.deg2rad(-4.0)])
R = slam.R

# Correct covariance (with cross-covariances)
mu_good, Sigma_good = ekf_update(slam.state.copy(), slam.covariance.copy(), z, z_hat, H, R)

# Incorrect covariance: manually zero robot<->landmark cross terms
Sigma_bad0 = slam.covariance.copy()
Sigma_bad0[:3, 3:] = 0.0
Sigma_bad0[3:, :3] = 0.0
mu_bad, Sigma_bad = ekf_update(slam.state.copy(), Sigma_bad0, z, z_hat, H, R)

print("Pose correction norm with proper cross-cov:", np.linalg.norm(mu_good[:3] - slam.state[:3]))
print("Pose correction norm with zeroed cross-cov:", np.linalg.norm(mu_bad[:3] - slam.state[:3]))

plt.figure(figsize=(5, 4))
plt.bar(
    ["proper cross-cov", "zeroed cross-cov"],
    [np.linalg.norm(mu_good[:3] - slam.state[:3]), np.linalg.norm(mu_bad[:3] - slam.state[:3])],
)
plt.ylabel("robot pose correction norm")
plt.title("Cross-covariances affect robot correction")
plt.tight_layout()
plt.show()


### Exercise 1
Try setting only part of the cross-covariance block to zero (for example just `x` terms) and compare the update.

<details>
<summary>Optional solution idea</summary>

Copy `Sigma_bad0` and zero only selected rows/columns, then rerun `ekf_update`.
Compare robot correction norm and updated covariance trace.

</details>


## 5) Exercise: sparse landmarks vs dense landmarks

We'll run short EKF-SLAM simulations with the same controls but different landmark counts.
This is a quick way to feel how map density influences state size and uncertainty trends.


In [ ]:
def run_short_world(n_landmarks, seed=2, steps=50):
    world = World2D.random(seed=seed, n_landmarks=n_landmarks, xlim=(-8, 8), ylim=(-8, 8))
    sim_cfg = SimConfig(
        dt=0.1,
        steps=steps,
        sensor_range=9.0,
        trajectory_mode="figure_eight",
        measurement_dropout_prob=0.0,
    )
    sim = Simulator2D(world=world, config=sim_cfg, seed=seed)
    local_slam = EKFSLAM(
        motion_model=DifferentialDriveMotionModel(),
        measurement_model=RangeBearingMeasurementModel(),
        config=EKFSLAMConfig(process_noise=sim.control_cov, measurement_noise=sim.measurement_cov),
    )

    for u, dt, measurements, _ in sim.run(steps):
        local_slam.step(u=u, dt=dt, measurements=measurements)

    return {
        "n_landmarks_in_world": n_landmarks,
        "n_landmarks_in_state": len(local_slam.landmark_states()),
        "state_dim": local_slam.state.size,
        "trace_pose_cov": float(np.trace(local_slam.covariance[:3, :3])),
        "trace_full_cov": float(np.trace(local_slam.covariance)),
    }

sparse_stats = run_short_world(n_landmarks=4, seed=10, steps=50)
dense_stats = run_short_world(n_landmarks=18, seed=10, steps=50)

print("Sparse world stats:", sparse_stats)
print("Dense world stats:", dense_stats)


### Exercise 2
- Change `sensor_range` and rerun sparse/dense experiments.
- Compare number of observed landmarks versus total landmarks in world.
- Plot covariance trace over time (instead of only final value).

<details>
<summary>Optional solution hint</summary>

Store `trace(local_slam.covariance[:3,:3])` at each step in `run_short_world` and plot it.

</details>


## 6) Final mini-run: EKFSLAM on a small world with printed stats

As requested, this section runs the repository EKF-SLAM implementation on a small world and prints:
- number of landmarks in state
- covariance shape
- innovation statistics


In [ ]:
world = World2D.random(seed=5, n_landmarks=6, xlim=(-7, 7), ylim=(-7, 7))
sim = Simulator2D(
    world=world,
    config=SimConfig(dt=0.1, steps=70, sensor_range=8.5, trajectory_mode="figure_eight"),
    seed=5,
)

slam_small = EKFSLAM(
    motion_model=DifferentialDriveMotionModel(),
    measurement_model=RangeBearingMeasurementModel(),
    config=EKFSLAMConfig(process_noise=sim.control_cov, measurement_noise=sim.measurement_cov),
)

for u, dt, measurements, _ in sim.run(70):
    slam_small.step(u=u, dt=dt, measurements=measurements)

n_landmarks_state = len(slam_small.landmark_states())
cov_shape = slam_small.covariance.shape
nis = np.array(slam_small.nis_values, dtype=float)

print("Number of landmarks in EKF state:", n_landmarks_state)
print("Covariance shape:", cov_shape)
if nis.size > 0:
    print("Innovation stats (NIS):")
    print("  count:", int(nis.size))
    print("  mean :", float(np.mean(nis)))
    print("  std  :", float(np.std(nis)))
    print("  max  :", float(np.max(nis)))
else:
    print("Innovation stats (NIS): no landmark updates occurred.")


## Wrap-up

Key takeaway:
- State augmentation is not just appending landmark coordinates.
- You must also append the right covariance blocks, especially cross-covariances.
- Those cross terms are what let future measurements correctly update both map and robot pose.
